# AI Replace Inpaint Smoke Runner

Standalone Pokecut-style AI Replace flow under inpaint/.

This notebook mirrors the main SD3.5/Add-it runners: install, clone/update, imports, runtime check, HF login, ready dataset paths, smoke run, metrics, previews, export.

## 1. Install Dependencies

In [ ]:
!pip install -q "diffusers>=0.30.0,<1.0.0" "transformers>=4.40.0" "accelerate>=0.30.0" safetensors ultralytics huggingface_hub opencv-python pillow numpy pandas matplotlib


## 2. Clone Or Update Repo

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/BDT-17/VIN.git"
REPO_DIR = Path("/kaggle/working/VIN")

if REPO_DIR.exists():
    %cd /kaggle/working/VIN
    !git fetch origin main
    !git pull --ff-only origin main
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd /kaggle/working/VIN

PROJECT_DIR = REPO_DIR if (REPO_DIR / "inpaint").exists() else Path.cwd()
print("PROJECT_DIR:", PROJECT_DIR)


## 3. Imports

In [ ]:
import os
import sys
import json
import shutil
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

PROJECT_DIR = Path("/kaggle/working/VIN") if Path("/kaggle/working/VIN/inpaint").exists() else Path.cwd()
sys.path = [str(PROJECT_DIR)] + [path for path in sys.path if path != str(PROJECT_DIR)]
%cd {PROJECT_DIR}

from inpaint.config import DEFAULT_CONFIG, AIReplaceConfig
from inpaint.smoke_runner import run_smoke, git_pull_ff_only, list_source_images

print("Loaded inpaint flow:", DEFAULT_CONFIG.AI_REPLACE_FLOW)
print("Model:", DEFAULT_CONFIG.MODEL_ID)


## 4. Runtime Check

In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu count:", torch.cuda.device_count())
    for idx in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(idx)
        print(idx, props.name, round(props.total_memory / 1024**3, 2), "GB")


## 5. Hugging Face Login

In [ ]:
import os
from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = hf_token or UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    print("No Kaggle HF_TOKEN secret found:", type(exc).__name__)

if hf_token:
    login(token=hf_token)
    print("Logged in to Hugging Face")
else:
    print("No HF token provided; public models only")


## 6. Ready Dataset Paths

Paths copied from sd35_run.ipynb / main config. Pick one dataset or set RUN_ALL_DATASETS=True.

In [ ]:
from pathlib import Path

DATASET_SMOKE_RUNS = {
    "citypersons_bg_yolo": {
        "input_dir": Path("/kaggle/input/datasets/muttahirulislam/citypersons-dataset-with-bg-image/yolo_dir/yolo_dir"),
        "num_images": 20,
    },
    "cityperson_nguyena": {
        "input_dir": Path("/kaggle/input/datasets/nguyenaabcxyzeric/cityperson"),
        "num_images": 20,
    },
    "mot17_02_frcnn": {
        "input_dir": Path("/kaggle/input/datasets/kyoru4444/mot17-02-fcrnn/MOT17-02-FRCNN"),
        "num_images": 20,
    },
    "human_detection_dataset": {
        "input_dir": Path("/kaggle/input/datasets/constantinwerner/human-detection-dataset/human detection dataset/0"),
        "num_images": 20,
    },
}

DATASET_CANDIDATES = [
    Path('/kaggle/input/datasets/kyoru4444/mot17-02-fcrnn/MOT17-02-FRCNN'),
    Path('/kaggle/input/datasets/kyoru4444/mot17-02-fcrnn/MOT17-02-FRCNN/img1'),
    Path('/kaggle/input/datasets/constantinwerner/human-detection-dataset/human detection dataset'),
    Path('/kaggle/input/datasets/constantinwerner/human-detection-dataset/human detection dataset/0'),
    Path('/kaggle/input/human-detection-dataset/human detection dataset'),
    Path('/kaggle/input/human-detection-dataset/human detection dataset/0'),
    Path('/kaggle/input/datasets/nguyenaabcxyzeric/cityperson'),
    Path('/kaggle/input/datasets/nguyenaabcxyzeric/CityPerson'),
    Path('/kaggle/input/datasets/nguyenaabcxyzeric/cityperson/test/images'),
    Path('/kaggle/input/datasets/nguyenaabcxyzeric/CityPerson/test/images'),
    Path('/kaggle/input/datasets/muttahirulislam/citypersons-dataset-with-bg-image/yolo_dir/yolo_dir'),
    Path('/kaggle/input/citypersons-dataset-with-bg-image/yolo_dir/yolo_dir'),
    Path('/kaggle/input/citypersons-dataset-with-bg-image'),
    Path('/kaggle/input/datasets/samyamine23/cityperson'),
    Path('/kaggle/input/cityperson'),
    Path('/kaggle/input/citypersons'),
    Path('/kaggle/input/city-persons'),
    Path('/kaggle/input/city-persons-2-0'),
    Path('/kaggle/input/city-persons-20'),
    Path('/kaggle/input/dataset'),
    Path('/kaggle/working/Dataset'),
]

SELECTED_DATASET = "citypersons_bg_yolo"  # citypersons_bg_yolo | cityperson_nguyena | mot17_02_frcnn | human_detection_dataset
RUN_ALL_DATASETS = False
NUM_IMAGES = 20
SEED = 42
DRY_RUN = False
USE_YOLO = True
OUTPUT_BASE = Path('/kaggle/working/ai_replace_smoke')

print('Configured datasets:')
for name, cfg in DATASET_SMOKE_RUNS.items():
    path = cfg['input_dir']
    print(f"  {name:24s} exists={path.exists()} path={path}")

existing_candidates = [path for path in DATASET_CANDIDATES if path.exists()]
print('Existing fallback candidates:', existing_candidates[:8])


## 7. Dataset Availability Check

In [ ]:
dataset_image_counts = {}
for name, cfg in DATASET_SMOKE_RUNS.items():
    input_dir = cfg['input_dir']
    if not input_dir.exists():
        dataset_image_counts[name] = 0
        continue
    dataset_image_counts[name] = len(list_source_images(input_dir, limit=cfg.get('num_images', NUM_IMAGES)))

dataset_image_counts


## 8. Optional Dry-Run Wiring Test

In [ ]:
selected_cfg = DATASET_SMOKE_RUNS[SELECTED_DATASET]
if selected_cfg['input_dir'].exists():
    dry_rows, dry_summary = run_smoke(
        input_dir=selected_cfg['input_dir'],
        output_dir=Path('/kaggle/working/ai_replace_dry_run') / SELECTED_DATASET,
        num_images=min(3, NUM_IMAGES),
        seed=SEED,
        load_model=False,
        use_yolo=False,
    )
    dry_summary
else:
    print('Selected dataset path does not exist:', selected_cfg['input_dir'])


## 9. Full Model Smoke Test

In [ ]:
def run_inpaint_dataset_smoke(dataset_name, smoke_images=None):
    cfg = DATASET_SMOKE_RUNS[dataset_name]
    input_dir = cfg['input_dir']
    if not input_dir.exists():
        raise FileNotFoundError(f'{dataset_name} path not found: {input_dir}')
    output_dir = OUTPUT_BASE / dataset_name
    rows, summary = run_smoke(
        input_dir=input_dir,
        output_dir=output_dir,
        num_images=smoke_images or cfg.get('num_images', NUM_IMAGES),
        seed=SEED,
        load_model=not DRY_RUN,
        use_yolo=USE_YOLO,
    )
    return rows, summary, output_dir

if RUN_ALL_DATASETS:
    all_smoke_results = {}
    for dataset_name in DATASET_SMOKE_RUNS:
        if DATASET_SMOKE_RUNS[dataset_name]['input_dir'].exists():
            all_smoke_results[dataset_name] = run_inpaint_dataset_smoke(dataset_name, smoke_images=NUM_IMAGES)
        else:
            print('SKIP missing dataset:', dataset_name, DATASET_SMOKE_RUNS[dataset_name]['input_dir'])
    rows, summary, OUTPUT_DIR = next(reversed(all_smoke_results.values())) if all_smoke_results else ([], {}, OUTPUT_BASE)
else:
    rows, summary, OUTPUT_DIR = run_inpaint_dataset_smoke(SELECTED_DATASET, smoke_images=NUM_IMAGES)

summary


## 10. Metrics Summary

In [ ]:
def load_metrics(output_dir):
    summary_path = Path(output_dir) / 'metrics' / 'metrics_summary.json'
    return json.load(open(summary_path, encoding='utf-8'))

if RUN_ALL_DATASETS:
    dataset_summaries = {name: load_metrics(result[2]) for name, result in all_smoke_results.items()}
    dataset_summaries
else:
    summary = load_metrics(OUTPUT_DIR)
    summary


## 11. Manifest Preview

In [ ]:
import pandas as pd

manifest_csv = Path(OUTPUT_DIR) / 'manifest.csv'
df = pd.read_csv(manifest_csv)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('rows:', len(df))
display_cols = [
    'accepted', 'reject_reason', 'outside_mask_diff', 'object_mask_inside_ratio',
    'opacity_score', 'background_preservation_score', 'ai_replace_quality_score',
    'source_path', 'output_path',
]
df[[col for col in display_cols if col in df.columns]].head(20)


## 12. Preview Gallery

In [ ]:
import math
import matplotlib.pyplot as plt
from PIL import Image

preview_dir = Path(OUTPUT_DIR) / 'previews'
harmonized = sorted(preview_dir.glob('*_harmonized.png'))[:6]
if not harmonized:
    print('No previews found:', preview_dir)
else:
    cols = 3
    rows_n = math.ceil(len(harmonized) / cols)
    plt.figure(figsize=(5 * cols, 5 * rows_n))
    for i, path in enumerate(harmonized, 1):
        plt.subplot(rows_n, cols, i)
        plt.imshow(Image.open(path))
        plt.title(path.name[:48])
        plt.axis('off')
    plt.tight_layout()


## 13. Outside-Mask Diff Preview

These should be nearly black when hard restore is working.

In [ ]:
diffs = sorted((Path(OUTPUT_DIR) / 'previews').glob('*_diff_outside_mask.png'))[:6]
if not diffs:
    print('No diff previews found')
else:
    cols = 3
    rows_n = math.ceil(len(diffs) / cols)
    plt.figure(figsize=(5 * cols, 5 * rows_n))
    for i, path in enumerate(diffs, 1):
        plt.subplot(rows_n, cols, i)
        plt.imshow(Image.open(path))
        plt.title(path.name[:48])
        plt.axis('off')
    plt.tight_layout()


## 14. Export Outputs

In [ ]:
zip_base = Path('/kaggle/working') / OUTPUT_BASE.name
zip_path = shutil.make_archive(str(zip_base), 'zip', root_dir=OUTPUT_BASE)
print('Saved export:', zip_path)
